<a href="https://colab.research.google.com/github/GinnaGomez09/proyecto_aplicado_javeriana/blob/main/notebooks/07_conversion_unidades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Clonar repositorio y preparar entorno
# ============================================================

!git clone https://github.com/GinnaGomez09/proyecto_aplicado_javeriana.git
%cd proyecto_aplicado_javeriana

Cloning into 'proyecto_aplicado_javeriana'...
remote: Enumerating objects: 442, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 442 (delta 142), reused 66 (delta 66), pack-reused 264 (from 2)
Receiving objects: 100% (442/442), 4.36 MiB | 7.14 MiB/s, done.
Resolving deltas: 100% (240/240), done.
/content/proyecto_aplicado_javeriana


In [2]:
# ============================================================
# 07_CONVERSION_UNIDADES.ipynb
# Conversión de unidades caseras a unidades métricas
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm

tqdm.pandas()

In [3]:
# ------------------------------------------------------------
# 1. Cargar dataset estandarizado
# ------------------------------------------------------------

input_csv = "data/interim/recetas_ingredientes_estandarizados.csv"

ingredientes = pd.read_csv(input_csv)

print("Dimensiones:", ingredientes.shape)
print("Número de recetas:", ingredientes["recipe_id"].nunique())

ingredientes.head()

Dimensiones: (3777, 15)
Número de recetas: 436


,recipe_id,recipe_title,ingredient_id,original_text,clean_text,ingredient_name_clean,unit_clean,quantity_original,quantity_value,unit_extracted,ingredient_extracted,preparation_state,metric_equivalent_value,metric_equivalent_unit,ingredient_standardized
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,1,1 taza de harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,taza,1,1.000000,taza,harina de arepa blanca o amarilla,NaN,NaN,NaN,harina de arepa
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,2,1 taza de agua tibia,1 taza de agua tibia,agua tibia,taza,1,1.000000,taza,agua,tibia,NaN,NaN,agua
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,3,⅓ taza de queso mozzarella o queso blanco rallado,1/3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,taza,1/3,0.333333,taza,queso mozzarella o queso blanco,rallado,NaN,NaN,queso mozzarella
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,4,2 cucharadas de mantequilla,2 cucharadas de mantequilla,mantequilla,cucharada,2,2.000000,cucharadas,mantequilla,NaN,NaN,NaN,mantequilla
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,5,Sal,sal,sal,NaN,1,1.000000,unidad,sal,NaN,NaN,NaN,sal


In [4]:
# ------------------------------------------------------------
# 2. Cargar JSON estandarizado
# ------------------------------------------------------------

input_json = "data/processed/recetas_estructura_json_estandarizada.json"

with open(input_json, "r", encoding="utf-8") as f:
    recetas_json = json.load(f)

print("Número de recetas en JSON:", len(recetas_json))

Número de recetas en JSON: 436


In [5]:
# ------------------------------------------------------------
# 3. Definir equivalencias generales de unidades
# ------------------------------------------------------------
# Estas equivalencias son aproximadas y deben ser refinadas
# según el tipo de ingrediente en etapas posteriores.

EQUIVALENCIAS_UNIDADES = {
    "gramo": ("g", 1),
    "gramos": ("g", 1),
    "g": ("g", 1),
    "gr": ("g", 1),

    "kilogramo": ("g", 1000),
    "kilogramos": ("g", 1000),
    "kg": ("g", 1000),

    "libra": ("g", 453.592),
    "libras": ("g", 453.592),
    "lb": ("g", 453.592),

    "onza": ("g", 28.3495),
    "onzas": ("g", 28.3495),
    "oz": ("g", 28.3495),

    "mililitro": ("ml", 1),
    "mililitros": ("ml", 1),
    "ml": ("ml", 1),

    "litro": ("ml", 1000),
    "litros": ("ml", 1000),
    "l": ("ml", 1000),

    "taza": ("ml", 240),
    "tazas": ("ml", 240),

    "cucharada": ("ml", 15),
    "cucharadas": ("ml", 15),

    "cucharadita": ("ml", 5),
    "cucharaditas": ("ml", 5)
}

In [6]:
# ------------------------------------------------------------
# 4. Equivalencias específicas por ingrediente
# ------------------------------------------------------------
# Cuando la unidad es casera, como taza o cucharada,
# la conversión a gramos depende del ingrediente.

DENSIDADES_APROXIMADAS = {
    "agua": 1.0,
    "leche": 1.03,
    "aceite vegetal": 0.92,
    "aceite de oliva": 0.92,
    "mantequilla": 0.96,
    "crema de leche": 1.0,

    "harina de trigo": 0.53,
    "harina de arepa": 0.55,
    "azucar": 0.85,
    "arroz": 0.80,
    "lenteja": 0.78,
    "frijol": 0.75,

    "queso": 0.60,
    "queso mozzarella": 0.60,
    "queso parmesano": 0.45,

    "cebolla": 0.65,
    "cebolla larga": 0.45,
    "tomate": 0.65,
    "cilantro": 0.20,
    "zanahoria": 0.65,
    "papa": 0.70,
    "yuca": 0.70,
    "platano": 0.65
}

In [7]:
# ------------------------------------------------------------
# 5. Pesos aproximados por unidad
# ------------------------------------------------------------
# Para ingredientes contados por unidad, diente, pieza, etc.

PESOS_POR_UNIDAD = {
    "ajo": {
        "diente": 3,
        "unidad": 3
    },
    "huevo": {
        "unidad": 50
    },
    "cebolla": {
        "unidad": 100
    },
    "tomate": {
        "unidad": 120
    },
    "papa": {
        "unidad": 150
    },
    "papa criolla": {
        "unidad": 40
    },
    "zanahoria": {
        "unidad": 70
    },
    "platano": {
        "unidad": 180
    },
    "pollo": {
        "pieza": 120,
        "piezas": 120,
        "unidad": 120
    }
}

In [8]:
# ------------------------------------------------------------
# 6. Función de conversión principal
# ------------------------------------------------------------

def convertir_a_metrico(row):
    cantidad = row["quantity_value"]
    unidad = row["unit_clean"] if pd.notna(row["unit_clean"]) else row["unit_extracted"]
    ingrediente = row["ingredient_standardized"]

    if pd.isna(cantidad):
        return np.nan, np.nan, "cantidad_no_disponible"

    unidad = str(unidad).lower().strip() if pd.notna(unidad) else "unidad"
    ingrediente = str(ingrediente).lower().strip() if pd.notna(ingrediente) else ""

    # 1. Si ya existe equivalencia métrica explícita, usarla
    if pd.notna(row.get("metric_equivalent_value")) and pd.notna(row.get("metric_equivalent_unit")):
        return (
            row["metric_equivalent_value"],
            row["metric_equivalent_unit"],
            "equivalencia_explicita"
        )

    # 2. Unidades métricas directas
    if unidad in EQUIVALENCIAS_UNIDADES:
        unidad_metrica, factor = EQUIVALENCIAS_UNIDADES[unidad]
        valor_convertido = cantidad * factor

        # Si queda en ml y existe densidad del ingrediente, convertir a gramos
        if unidad_metrica == "ml" and ingrediente in DENSIDADES_APROXIMADAS:
            valor_g = valor_convertido * DENSIDADES_APROXIMADAS[ingrediente]
            return valor_g, "g", "conversion_volumen_a_gramos_por_densidad"

        return valor_convertido, unidad_metrica, "conversion_directa"

    # 3. Conversión por unidad específica de ingrediente
    if ingrediente in PESOS_POR_UNIDAD:
        reglas = PESOS_POR_UNIDAD[ingrediente]

        if unidad in reglas:
            valor_g = cantidad * reglas[unidad]
            return valor_g, "g", "conversion_por_peso_unitario"

    # 4. Si no se puede convertir
    return np.nan, np.nan, "sin_conversion"

In [9]:
# ------------------------------------------------------------
# 7. Aplicar conversión de unidades
# ------------------------------------------------------------

ingredientes_convertidos = ingredientes.copy()

resultado_conversion = ingredientes_convertidos.progress_apply(
    convertir_a_metrico,
    axis=1
)

(
    ingredientes_convertidos["metric_quantity_value"],
    ingredientes_convertidos["metric_quantity_unit"],
    ingredientes_convertidos["conversion_method"]
) = zip(*resultado_conversion)

ingredientes_convertidos.head()

100%|██████████| 3777/3777 [00:00<00:00, 23830.27it/s]


,recipe_id,recipe_title,ingredient_id,original_text,clean_text,ingredient_name_clean,unit_clean,quantity_original,quantity_value,unit_extracted,ingredient_extracted,preparation_state,metric_equivalent_value,metric_equivalent_unit,ingredient_standardized,metric_quantity_value,metric_quantity_unit,conversion_method
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,1,1 taza de harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,taza,1,1.000000,taza,harina de arepa blanca o amarilla,NaN,NaN,NaN,harina de arepa,132.0,g,conversion_volumen_a_gramos_por_densidad
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,2,1 taza de agua tibia,1 taza de agua tibia,agua tibia,taza,1,1.000000,taza,agua,tibia,NaN,NaN,agua,240.0,g,conversion_volumen_a_gramos_por_densidad
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,3,⅓ taza de queso mozzarella o queso blanco rallado,1/3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,taza,1/3,0.333333,taza,queso mozzarella o queso blanco,rallado,NaN,NaN,queso mozzarella,48.0,g,conversion_volumen_a_gramos_por_densidad
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,4,2 cucharadas de mantequilla,2 cucharadas de mantequilla,mantequilla,cucharada,2,2.000000,cucharadas,mantequilla,NaN,NaN,NaN,mantequilla,28.8,g,conversion_volumen_a_gramos_por_densidad
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,5,Sal,sal,sal,NaN,1,1.000000,unidad,sal,NaN,NaN,NaN,sal,NaN,NaN,sin_conversion


In [10]:
# ------------------------------------------------------------
# 8. Validación rápida de conversiones
# ------------------------------------------------------------

print("Métodos de conversión:")
display(
    ingredientes_convertidos["conversion_method"]
    .value_counts(dropna=False)
)

print("Unidades métricas resultantes:")
display(
    ingredientes_convertidos["metric_quantity_unit"]
    .value_counts(dropna=False)
)

Métodos de conversión:


,count
conversion_method,
conversion_directa,1427
sin_conversion,1046
conversion_volumen_a_gramos_por_densidad,926
conversion_por_peso_unitario,378


Unidades métricas resultantes:


,count
metric_quantity_unit,
g,1525
ml,1206
NaN,1046


In [11]:
# ------------------------------------------------------------
# 9. Revisar ejemplos convertidos
# ------------------------------------------------------------

display(
    ingredientes_convertidos[
        [
            "original_text",
            "ingredient_standardized",
            "quantity_value",
            "unit_extracted",
            "unit_clean",
            "metric_quantity_value",
            "metric_quantity_unit",
            "conversion_method"
        ]
    ].sample(25, random_state=42)
)

,original_text,ingredient_standardized,quantity_value,unit_extracted,unit_clean,metric_quantity_value,metric_quantity_unit,conversion_method
3530,1 lata de leche condensada,leche condensada,1.00,lata,lata,NaN,NaN,sin_conversion
999,2 cucharadas de aceite vegetal,aceite vegetal,2.00,cucharadas,cucharada,27.600,g,conversion_volumen_a_gramos_por_densidad
3023,1 taza de queso mozzarella cortado en cubitos,queso mozzarella,1.00,taza,taza,144.000,g,conversion_volumen_a_gramos_por_densidad
1550,2 hojas de laurel,laurel,2.00,hojas,hoja,NaN,NaN,sin_conversion
2428,3 tazas de fresas frescas lavadas y cortadas p...,fresa y por,3.00,tazas,taza,720.000,ml,conversion_directa
3732,2 cucharadas de vinagre blanco,vinagre,2.00,cucharadas,cucharada,30.000,ml,conversion_directa
2955,1 taza de fresas cortadas en cubitos,fresa,1.00,taza,taza,240.000,ml,conversion_directa
3646,1 cucharada de extracto de vainilla,extracto de vainilla,1.00,cucharada,cucharada,15.000,ml,conversion_directa
465,1 cucharada de aceite,aceite,1.00,cucharada,cucharada,15.000,ml,conversion_directa
1123,2 cucharadas de leche,leche,2.00,cucharadas,cucharada,30.900,g,conversion_volumen_a_gramos_por_densidad


In [12]:
# ------------------------------------------------------------
# 10. Revisar ingredientes sin conversión
# ------------------------------------------------------------

sin_conversion = ingredientes_convertidos[
    ingredientes_convertidos["conversion_method"] == "sin_conversion"
].copy()

print("Registros sin conversión:", sin_conversion.shape[0])

display(
    sin_conversion[
        [
            "original_text",
            "ingredient_standardized",
            "quantity_value",
            "unit_extracted",
            "unit_clean"
        ]
    ].head(50)
)

Registros sin conversión: 1046


,original_text,ingredient_standardized,quantity_value,unit_extracted,unit_clean
4,Sal,sal,1.0,unidad,NaN
19,1 pimientón rojo finamente picado,pimienton rojo,1.0,unidad,NaN
22,Un cuarto de cucharadita de achiote,achiote,1.0,unidad,NaN
23,3 mazorcas de maíz cortado en 3 piezas,mazorca de maiz en 3 pieza,3.0,unidad,NaN
31,Un cuarto de taza de cilantro fresco picado,cilantro,1.0,unidad,NaN
32,Un cuarto de cucharadita de pimienta molida,cuarto de cucharadita de pimienta,1.0,unidad,NaN
34,"1 posta o muchacho de 3 a 4 libras, cortada en...",posta o muchacho de 3 a 4 libra gruesa,1.0,unidad,NaN
41,Sal y pimienta al gusto,sal y pimienta,1.0,unidad,NaN
44,Cilantro fresco,cilantro,1.0,unidad,NaN
64,2 cebollas largas picadas,cebolla larga,2.0,unidad,NaN


In [13]:
# ------------------------------------------------------------
# 11. Actualizar JSON con cantidades métricas
# ------------------------------------------------------------

mapa_conversion = {}

for _, row in ingredientes_convertidos.iterrows():
    clave = (
        str(row["recipe_id"]),
        int(row["ingredient_id"])
    )

    mapa_conversion[clave] = {
        "value": None if pd.isna(row["metric_quantity_value"]) else float(row["metric_quantity_value"]),
        "unit": None if pd.isna(row["metric_quantity_unit"]) else row["metric_quantity_unit"],
        "method": row["conversion_method"]
    }


for receta in recetas_json:
    recipe_id = str(receta["recipe_id"])

    for ingrediente in receta["ingredients"]:
        ingredient_id = ingrediente["ingredient_id"]
        clave = (recipe_id, ingredient_id)

        if clave in mapa_conversion:
            ingrediente["metric_quantity"] = mapa_conversion[clave]

    receta["pipeline_stage"] = "07_conversion_unidades"
    receta["structure_version"] = "1.2"

print("JSON actualizado correctamente.")

JSON actualizado correctamente.


In [14]:
# ------------------------------------------------------------
# 12. Visualizar ejemplo actualizado
# ------------------------------------------------------------

print(json.dumps(recetas_json[0], ensure_ascii=False, indent=2))

{
  "recipe_id": "86af61e4-e16a-11ed-9591-a96d6180cd25",
  "title": "Arepas de Queso",
  "pipeline_stage": "07_conversion_unidades",
  "structure_version": "1.2",
  "ingredients": [
    {
      "ingredient_id": 1,
      "original_text": "1 taza de harina de arepa blanca o amarilla",
      "clean_text": "1 taza de harina de arepa blanca o amarilla",
      "quantity": {
        "original": "1",
        "value": 1.0
      },
      "unit": {
        "extracted": "taza",
        "clean": "taza"
      },
      "ingredient": {
        "raw": "harina de arepa blanca o amarilla",
        "clean_reference": "harina de arepa blanca o amarilla",
        "standardized": "harina de arepa"
      },
      "preparation_state": null,
      "metric_equivalent": {
        "value": null,
        "unit": null
      },
      "tcac_match": null,
      "metric_quantity": {
        "value": 132.0,
        "unit": "g",
        "method": "conversion_volumen_a_gramos_por_densidad"
      }
    },
    {
      "ingre

In [15]:
# ------------------------------------------------------------
# 13. Guardar salidas
# ------------------------------------------------------------

Path("data/interim").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("reports/tables").mkdir(parents=True, exist_ok=True)

output_csv = "data/interim/recetas_unidades_convertidas.csv"
output_json = "data/processed/recetas_estructura_json_convertida.json"
output_sin_conversion = "reports/tables/ingredientes_sin_conversion_unidades.csv"

ingredientes_convertidos.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(
        recetas_json,
        f,
        ensure_ascii=False,
        indent=2
    )

sin_conversion.to_csv(
    output_sin_conversion,
    index=False,
    encoding="utf-8-sig"
)

print("Archivos guardados:")
print(output_csv)
print(output_json)
print(output_sin_conversion)

Archivos guardados:
data/interim/recetas_unidades_convertidas.csv
data/processed/recetas_estructura_json_convertida.json
reports/tables/ingredientes_sin_conversion_unidades.csv


# Conversión de unidades

En esta etapa se convierten las cantidades extraídas a unidades métricas, principalmente gramos o mililitros, para facilitar la integración posterior con la TCAC.

La TCAC expresa la composición nutricional de los alimentos generalmente por cada 100 gramos, por lo que es necesario transformar las unidades caseras de las recetas, como tazas, cucharadas, cucharaditas, libras o unidades, a una medida métrica aproximada.

Esta etapa toma como entrada:

`data/interim/recetas_ingredientes_estandarizados.csv`

y actualiza también la estructura JSON estandarizada:

`data/processed/recetas_estructura_json_estandarizada.json`

In [16]:
from google.colab import files

files.download("data/interim/recetas_unidades_convertidas.csv")
files.download("data/processed/recetas_estructura_json_convertida.json")
files.download("reports/tables/ingredientes_sin_conversion_unidades.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>